# 01 — Exploratory Data Analysis
**PIX Fraud Intelligence | IBM Portfolio Project**

Dataset: [Credit Card Fraud Detection](https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud) (Kaggle)  
We adapt it to simulate PIX instant payment transactions in the Brazilian market.

**Goals:**
- Understand class imbalance (fraud rate ~0.17%)
- Explore feature distributions and correlations
- Identify the most discriminative PCA components
- Generate visualizations for the README

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
sns.set_palette('husl')

## 1. Load Data
Download `creditcard.csv` from Kaggle and place it in `data/raw/`.

In [ ]:
df = pd.read_csv('../data/raw/creditcard.csv')
print(f'Shape: {df.shape}')
df.head()

In [ ]:
# Sample to 100k rows — fits comfortably in DB2 Lite (200MB)
df_sample = df.sample(n=100_000, random_state=42, weights=df['Class'].map({0: 1, 1: 50}))
df_sample = df_sample.reset_index(drop=True)
print(f'Sampled shape: {df_sample.shape}')
print(f'Fraud rate: {df_sample["Class"].mean():.4%}')

## 2. Class Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

counts = df_sample['Class'].value_counts()
axes[0].bar(['Legítima', 'Fraude'], counts.values, color=['#0062ff', '#ff5c49'])
axes[0].set_title('Distribuição de Classes')
axes[0].set_ylabel('Número de transações')
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 100, f'{v:,}', ha='center', fontweight='bold')

axes[1].pie(
    counts.values,
    labels=['Legítima', 'Fraude'],
    autopct='%1.2f%%',
    colors=['#0062ff', '#ff5c49'],
    startangle=90,
    wedgeprops={'edgecolor': 'white', 'linewidth': 2}
)
axes[1].set_title('Proporção de Classes')

plt.suptitle('Desbalanceamento de Classes — Transações PIX', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../assets/class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Transaction Amount Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for label, color, name in [(0, '#0062ff', 'Legítima'), (1, '#ff5c49', 'Fraude')]:
    axes[0].hist(
        np.log1p(df_sample[df_sample['Class'] == label]['Amount']),
        bins=50, alpha=0.6, color=color, label=name
    )
axes[0].set_title('Distribuição log(Valor + 1) por Classe')
axes[0].set_xlabel('log(Amount + 1)')
axes[0].legend()

df_sample['hour'] = (df_sample['Time'] // 3600) % 24
fraud_by_hour = df_sample.groupby('hour')['Class'].mean().reset_index()
axes[1].plot(fraud_by_hour['hour'], fraud_by_hour['Class'] * 100, marker='o', color='#ff5c49', linewidth=2)
axes[1].set_title('Taxa de Fraude por Hora do Dia')
axes[1].set_xlabel('Hora')
axes[1].set_ylabel('Taxa de Fraude (%)')
axes[1].set_xticks(range(0, 24))

plt.suptitle('Comportamento das Transações', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../assets/amount_time_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Most Discriminative PCA Features

In [ ]:
pca_cols = [f'V{i}' for i in range(1, 29)]

fraud_means = df_sample[df_sample['Class'] == 1][pca_cols].mean()
legit_means = df_sample[df_sample['Class'] == 0][pca_cols].mean()
diff = (fraud_means - legit_means).abs().sort_values(ascending=False)

top10 = diff.head(10)
fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.bar(top10.index, top10.values, color='#0062ff')
ax.set_title('Top 10 Features — Diferença de Média entre Fraude e Legítima', fontweight='bold')
ax.set_ylabel('|mean(fraude) - mean(legítima)|')
for bar, val in zip(bars, top10.values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01, f'{val:.2f}', ha='center', fontsize=9)
plt.tight_layout()
plt.savefig('../assets/top_features.png', dpi=150, bbox_inches='tight')
plt.show()
print('Top 10 features mais discriminativas:', top10.index.tolist())

## 5. Correlation Heatmap (Top Features)

In [ ]:
top_feats = top10.index.tolist() + ['Amount', 'Class']
corr = df_sample[top_feats].corr()

fig, ax = plt.subplots(figsize=(12, 9))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, ax=ax, linewidths=0.5)
ax.set_title('Correlação — Top Features + Class', fontweight='bold')
plt.tight_layout()
plt.savefig('../assets/correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Save Sampled Dataset

In [ ]:
import os
os.makedirs('../data/raw', exist_ok=True)
df_sample.to_csv('../data/raw/transactions_sampled.csv', index=False)
print(f'Saved {len(df_sample):,} rows to data/raw/transactions_sampled.csv')
print(f'File size: {os.path.getsize("../data/raw/transactions_sampled.csv") / 1e6:.1f} MB')